In [ ]:
# data_ingest_sqlite.py

import sqlite3
import pandas as pd
from pathlib import Path

CSV_PATH = Path("resqfood_listings.csv")
DB_PATH = Path("resqfood.db")

# -----------------------------
# VALIDATION
# -----------------------------
if not CSV_PATH.exists():
    raise FileNotFoundError(f"{CSV_PATH} not found")

print("Loading CSV...")
df = pd.read_csv(CSV_PATH)

# Required columns
required_cols = [
    "listing_id", "donor_id", "donor_type", "food_type",
    "quantity", "unit", "city", "packaging",
    "time_posted", "time_since_cooked_mins",
    "donor_reliability", "distance_km",
    "ambient_temp_c", "freshness_score",
    "pickup_prob", "picked_up"
]

missing = [c for c in required_cols if c not in df.columns]
if missing:
    raise ValueError(f"Missing columns: {missing}")

# -----------------------------
# CLEANING
# -----------------------------
df = df.drop_duplicates(subset=["listing_id"])

df["time_posted"] = pd.to_datetime(df["time_posted"], errors="coerce")

# Remove invalid rows
df = df.dropna(subset=["time_posted"])

# Convert to ISO format (important for SQLite)
df["time_posted"] = df["time_posted"].dt.strftime("%Y-%m-%d %H:%M:%S")

# Fill numeric NaNs safely
numeric_cols = [
    "distance_km", "ambient_temp_c",
    "freshness_score", "pickup_prob"
]

for col in numeric_cols:
    df[col] = df[col].fillna(df[col].median())

print(f"Final rows after cleaning: {len(df)}")

# -----------------------------
# SQLITE INSERT
# -----------------------------
with sqlite3.connect(DB_PATH) as conn:
    cursor = conn.cursor()

    # Create table with schema
    cursor.execute("""
    CREATE TABLE IF NOT EXISTS listings (
        listing_id TEXT PRIMARY KEY,
        donor_id TEXT,
        donor_type TEXT,
        food_type TEXT,
        quantity REAL,
        unit TEXT,
        city TEXT,
        packaging TEXT,
        time_posted TEXT,
        time_since_cooked_mins INTEGER,
        donor_reliability TEXT,
        distance_km REAL,
        ambient_temp_c REAL,
        freshness_score REAL,
        pickup_prob REAL,
        picked_up INTEGER
    )
    """)

    # Clear old data
    cursor.execute("DELETE FROM listings")

    # Insert data
    df.to_sql("listings", conn, if_exists="append", index=False)

    # -----------------------------
    # INDEXES (VERY IMPORTANT)
    # -----------------------------
    cursor.execute("CREATE INDEX IF NOT EXISTS idx_time ON listings(time_posted)")
    cursor.execute("CREATE INDEX IF NOT EXISTS idx_city ON listings(city)")
    cursor.execute("CREATE INDEX IF NOT EXISTS idx_pickup ON listings(picked_up)")

    conn.commit()

print(f"✅ Data successfully ingested into {DB_PATH}")

Ingested into resqfood.db
